# Dwellsy Data Preparation

Prepare the complete Chicago Dwellsy export and save the reusable dataset for modeling and geospatial analysis.

**Data:** real (Dwellsy)

**Inputs:** raw_file("chicago")

**Outputs:** chicago_prepared_full.csv

**Run after:** 01_data_exploration.ipynb

In [1]:
# No user-adjustable parameters are defined for this notebook.


In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np


ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "libs").is_dir())
sys.path.insert(0, str(ROOT))   # lets the notebook import the shared libs/ package

from libs import paths
from libs.real_data import raw_file



# 09. Dwellsy Data Preparation for Modeling

## 1. Introduction

This notebook prepares the complete Chicago Dwellsy dataset for rental price modeling.

The dataset contains 545,312 historical rental listing records and 47 original variables. Building on the initial exploration conducted in `08_dwellsy_data_exploration.ipynb`, we perform basic data quality checks and prepare a reusable dataset while preserving all original variables.

The main objectives are to:

- Load the complete Chicago dataset.
- Examine repeated observations and rental unit identifiers.
- Perform basic data quality checks.
- Create temporal features from listing creation dates.
- Save a complete prepared dataset for subsequent modeling and geospatial analysis.

Feature selection, missing value imputation, categorical encoding, and train-test splitting will be performed in subsequent modeling notebooks to minimize data leakage.



## 2. Data Loading and Initial Validation

### 2.1 Loading the Complete Chicago Dataset

We load all 47 original variables from the Chicago Dwellsy dataset, preserving the complete information available for subsequent analysis.

In the previous notebook, a random sample of 99,979 Chicago records was used for preliminary exploration. Here, we use all 545,312 available records.

Retaining all original variables allows us to evaluate additional predictors and modeling approaches without repeatedly reconstructing the dataset.


In [3]:
chicago_path = raw_file("chicago")

# Load all columns from the original Chicago dataset
chicago_df = pd.read_csv(
    chicago_path,
    dtype={"address_zip": "string"},
    low_memory=False
)

# Store original column names for later validation
original_columns = chicago_df.columns.tolist()

# Check dataset dimensions
print("Chicago dataset shape:", chicago_df.shape)

# Display all column names
print("\nAvailable columns:")
print(original_columns)

# Preview data
display(chicago_df.head())


Chicago dataset shape: (545312, 47)

Available columns:
['dwellsy_address_id', 'address1_id', 'address2_id', 'data_source_external_id', 'listing_id', 'community_id', 'community_name', 'company_name', 'top_down_community_count', 'top_down_address1_count', 'address_1', 'address_2', 'address_city', 'address_state', 'address_zip', 'address_country', 'address_type', 'latitude', 'longitude', 'msa_code', 'msa_name', 'year_built', 'bedrooms', 'full_baths', 'half_baths', 'square_feet', 'rent_amount', 'listing_deposit', 'property_listing_status', 'description', 'creation_time', 'deactivation_time', 'ss_raw_primary_number', 'ss_raw_street_predirection', 'ss_raw_street_name', 'ss_raw_street_suffix', 'ss_raw_street_postdirection', 'ss_raw_secondary_designator', 'ss_raw_secondary_number', 'ss_raw_default_city_name', 'ss_raw_state_abbreviation', 'ss_raw_zipcode', 'ss_raw_plus4_code', 'amenities', 'photos', 'uru_id', 'uru_type']


,dwellsy_address_id,address1_id,address2_id,data_source_external_id,listing_id,community_id,community_name,company_name,top_down_community_count,top_down_address1_count,...,ss_raw_secondary_designator,ss_raw_secondary_number,ss_raw_default_city_name,ss_raw_state_abbreviation,ss_raw_zipcode,ss_raw_plus4_code,amenities,photos,uru_id,uru_type
0,133822-8163780,133822,8163780.0,1739819,3675156,279577,6211 N Kenmore Ave,Mihail Marinov,28,28,...,Apt,501,Chicago,IL,60660,2151,12 Month Lease or Longer; Cable Ready; Cats OK...,https://media.dwellsy.com/prod/c151dd5f3919432...,542230,Apartment
1,129521-8112041,129521,8112041.0,3871623,3558998,6069979,Mode Logan Square,Jessica Goodnight,296,218,...,Apt,807,Chicago,IL,60642,6024,12 Month Lease or Longer; Air Conditioning; Ca...,https://media.dwellsy.com/prod/8c186d97caa2102...,1362919,Apartment
2,123813-8066896,123813,8066896.0,14582015,16668829,279329,21st Street Lofts,3L Living,137,137,...,Unit,433,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610,Apartment
3,123813-8066896,123813,8066896.0,14582015,19878436,279329,21st Street Lofts,3L Living,137,137,...,Unit,433,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610,Apartment
4,130630-8121250,130630,8121250.0,10139825,10639335,5764,2420 N Kedzie Blvd,Sam,54,54,...,Apt,206,Chicago,IL,60647,5740,Cats OK; Dishwasher; Gas included in rent; Har...,https://media.dwellsy.com/prod/cd5a80b0a1d4a17...,428523,Apartment



### 2.2 Examination of Listing and Rental Unit Identifiers

Before preparing the modeling dataset, we examine the structure of the listing and rental unit identifiers.

The variable `listing_id` identifies rental listings, while `uru_id` is used to identify rental units.

Since the dataset contains historical records, the same rental unit may appear in multiple listings or at different points in time.

We examine the number of unique identifiers, repeated observations, and missing values to better understand the dataset's structure.

These checks will help inform our data preparation strategy and reduce potential data leakage during model evaluation.


In [4]:
# Examine listing and rental unit identifiers
print("Total observations:", f"{len(chicago_df):,}")

print("\nListing identifiers:")
print("Unique listing_id:", f"{chicago_df['listing_id'].nunique():,}")
print("Repeated listing_id:", f"{chicago_df['listing_id'].duplicated().sum():,}")
print("Missing listing_id:", f"{chicago_df['listing_id'].isna().sum():,}")

print("\nRental unit identifiers:")
print("Unique uru_id:", f"{chicago_df['uru_id'].nunique():,}")
print("Repeated uru_id:", f"{chicago_df['uru_id'].duplicated().sum():,}")
print("Missing uru_id:", f"{chicago_df['uru_id'].isna().sum():,}")

# Check fully duplicated records
print("\nFully duplicated records:",
      f"{chicago_df.duplicated().sum():,}")


Total observations: 545,312

Listing identifiers:
Unique listing_id: 545,312
Repeated listing_id: 0
Missing listing_id: 0

Rental unit identifiers:
Unique uru_id: 152,233
Repeated uru_id: 393,079
Missing uru_id: 0



Fully duplicated records: 0



**Initial Findings**

The Chicago dataset contains 545,312 observations, each associated with a unique `listing_id`. No missing listing identifiers or fully duplicated records were identified.

In contrast, the dataset contains 152,233 unique `uru_id` values, indicating that multiple historical listing records may be associated with the same rental unit identifier.

On average, there are approximately 3.58 records per unique `uru_id`. However, this average does not describe the distribution of records across individual units.

These findings suggest that repeated unit identifiers are an important characteristic of the dataset and should not automatically be treated as duplicate observations.

For preliminary model evaluation, we plan to use a group-aware train-test split based on `uru_id` to prevent records associated with the same unit identifier from appearing in both training and test sets.

Further examination of listing dates and rental prices will help clarify the historical structure of repeated unit records.



### 2.3 Historical Records and Rental Price Variation

To better understand repeated rental unit identifiers, we examine whether records associated with the same `uru_id` have different listing dates and rental prices.

This analysis helps distinguish repeated unit identifiers from exact duplicate observations and provides preliminary evidence about the historical structure of the dataset.

We focus on a small number of units with multiple records rather than conducting an exhaustive longitudinal analysis.


In [5]:
# Convert listing creation dates to datetime
chicago_df["creation_time"] = pd.to_datetime(
    chicago_df["creation_time"],
    errors="coerce"
)

# Identify rental units with multiple records
unit_counts = chicago_df["uru_id"].value_counts()

print("Units with multiple records:",
      f"{(unit_counts > 1).sum():,}")

print("Maximum records for a single unit:",
      f"{unit_counts.max():,}")

# Select three units with multiple records
example_units = unit_counts[unit_counts > 1].head(3).index

# Examine historical listings for these units
example_records = chicago_df.loc[
    chicago_df["uru_id"].isin(example_units),
    ["uru_id", "listing_id", "creation_time", "rent_amount"]
].sort_values(["uru_id", "creation_time"])

display(example_records.groupby("uru_id").head(5))


Units with multiple records: 91,775
Maximum records for a single unit: 136


,uru_id,listing_id,creation_time,rent_amount
90621,1307167,3395355,2021-01-26 16:47:22,1727.0
139592,1307167,3491265,2021-02-08 11:13:27,1727.0
170530,1307167,3513754,2021-02-11 00:15:25,1727.0
94335,1307167,3840284,2021-03-19 15:04:25,1727.0
170531,1307167,3874427,2021-03-24 11:34:56,1727.0
6318,2107000,6206926,2021-12-30 19:59:47,1620.0
134149,2107000,6207759,2021-12-30 21:14:45,1620.0
134150,2107000,6329471,2022-01-14 03:08:47,1620.0
6319,2107000,6329472,2022-01-14 03:08:48,1620.0
134151,2107000,6348320,2022-01-15 01:09:33,1620.0


In [6]:
# Examine property characteristics for selected repeated units
selected_units = [1307167, 2107000, 6153001]

columns_to_inspect = [
    "uru_id",
    "listing_id",
    "creation_time",
    "rent_amount",
    "bedrooms",
    "full_baths",
    "square_feet",
    "year_built",
    "address_zip",
    "uru_type"
]

unit_details = chicago_df.loc[
    chicago_df["uru_id"].isin(selected_units),
    columns_to_inspect
].sort_values(["uru_id", "creation_time"])

# Display the first 10 records for each unit
display(unit_details.groupby("uru_id").head(10))


,uru_id,listing_id,creation_time,rent_amount,bedrooms,full_baths,square_feet,year_built,address_zip,uru_type
90621,1307167,3395355,2021-01-26 16:47:22,1727.0,1,1.0,955.0,2010.0,60616,Apartment
139592,1307167,3491265,2021-02-08 11:13:27,1727.0,1,1.0,955.0,2010.0,60616,Apartment
170530,1307167,3513754,2021-02-11 00:15:25,1727.0,1,1.0,955.0,2010.0,60616,Apartment
94335,1307167,3840284,2021-03-19 15:04:25,1727.0,1,1.0,955.0,2010.0,60616,Apartment
170531,1307167,3874427,2021-03-24 11:34:56,1727.0,1,1.0,955.0,2010.0,60616,Apartment
181285,1307167,3919908,2021-03-27 14:32:45,1727.0,1,1.0,955.0,2010.0,60616,Apartment
90622,1307167,3920456,2021-03-27 17:28:43,1727.0,1,1.0,955.0,2010.0,60616,Apartment
31762,1307167,3971532,2021-04-02 09:22:20,1727.0,1,1.0,955.0,2010.0,60616,Apartment
159837,1307167,4087035,2021-04-16 19:38:57,1727.0,1,1.0,955.0,2010.0,60616,Apartment
21873,1307167,4123971,2021-04-21 22:05:57,1727.0,1,1.0,955.0,2010.0,60616,Apartment


In [7]:
# Examine repeated units with different rental prices

unit_summary = chicago_df.groupby("uru_id").agg(
    n_listings=("listing_id", "size"),
    n_unique_rents=("rent_amount", "nunique"),
    min_rent=("rent_amount", "min"),
    max_rent=("rent_amount", "max")
)

multiple_listings = unit_summary["n_listings"] > 1

different_rents = (
    multiple_listings &
    (unit_summary["n_unique_rents"] > 1)
)

print(
    "Units with multiple listings:",
    f"{multiple_listings.sum():,}"
)

print(
    "Units with multiple listings and different rents:",
    f"{different_rents.sum():,}"
)

print(
    "Percentage of repeated units with different rents:",
    f"{100 * different_rents.sum() / multiple_listings.sum():.2f}%"
)

print(
    "Listings belonging to units with different rents:",
    f"{unit_summary.loc[different_rents, 'n_listings'].sum():,}"
)


Units with multiple listings: 91,775
Units with multiple listings and different rents: 73,721
Percentage of repeated units with different rents: 80.33%
Listings belonging to units with different rents: 426,951


In [8]:
# Identify simultaneous listings with different rental prices

simultaneous_listings = (
    chicago_df
    .groupby(["uru_id", "creation_time"])
    .agg(
        n_listings=("listing_id", "size"),
        n_unique_rents=("rent_amount", "nunique")
    )
)

simultaneous_conflicts = simultaneous_listings[
    (simultaneous_listings["n_listings"] > 1) &
    (simultaneous_listings["n_unique_rents"] > 1)
]

print(
    "Unit-timestamp groups with different rents:",
    f"{len(simultaneous_conflicts):,}"
)

print(
    "Listings in these groups:",
    f"{simultaneous_conflicts['n_listings'].sum():,}"
)

print(
    "Percentage of all Chicago listings:",
    f"{100 * simultaneous_conflicts['n_listings'].sum() / len(chicago_df):.2f}%"
)


Unit-timestamp groups with different rents: 641
Listings in these groups: 1,448
Percentage of all Chicago listings: 0.27%



**Initial Findings**

The Chicago dataset contains 152,233 unique rental unit identifiers (`uru_id`), of which 91,775 are associated with multiple listing records. The maximum number of listings associated with a single rental unit is 136.

According to the Dwellsy Data Dictionary, `uru_id` is a permanent unique identifier assigned to a rentable unit.

Among units with multiple listings, 73,721 (80.33%) have more than one recorded rental price. These units account for 426,951 listing records. Such variation may reflect legitimate changes in rental prices over time.

An additional examination identified 641 unit-timestamp groups containing different rental prices, involving 1,448 listings (0.27% of the Chicago dataset). Although these cases warrant further investigation, they represent a relatively small proportion of the available records.

Repeated listings may also result in some rental units receiving greater weight during model training and evaluation.

**Preliminary Modeling Decisions**

- Retain historical listing records without automatically removing observations that share the same `uru_id`.
- Use `uru_id` to create group-aware training and test sets, preventing the same rental unit from appearing in both.
- Consider temporal features derived from `creation_time` to account for changes in rental market conditions.
- Revisit repeated listings and potential inconsistencies during subsequent model refinement.

These decisions allow us to proceed with preliminary modeling while documenting limitations that may require further investigation.



## 3. Basic Data Quality Checks

Before preparing the final modeling dataset, we perform basic data quality checks on the complete Chicago dataset.

The objective is to identify missing values, invalid observations, and potential inconsistencies that could affect preliminary rental price prediction models.

We focus on the selected modeling variables and avoid extensive exploratory analysis, which is outside the scope of this notebook.

Missing value imputation and categorical encoding will be performed within the modeling pipelines to minimize data leakage.


### 3.1 Missing Values Assessment

We examine the number and percentage of missing values across all variables in the complete Chicago dataset.

This assessment provides an overview of data completeness and helps identify variables that may require additional preprocessing or investigation during subsequent modeling.


In [9]:
# Assess missing values in the complete Chicago dataset

missing_summary = pd.DataFrame({
    "Missing Count": chicago_df.isna().sum(),
    "Missing Percentage": chicago_df.isna().mean() * 100
})

missing_summary = missing_summary.sort_values(
    "Missing Percentage",
    ascending=False
)

display(missing_summary.round(2))


,Missing Count,Missing Percentage
ss_raw_street_postdirection,541741,99.35
half_baths,450510,82.62
listing_deposit,214240,39.29
year_built,176858,32.43
ss_raw_street_predirection,112096,20.56
ss_raw_secondary_number,52105,9.56
address2_id,52104,9.55
ss_raw_secondary_designator,52091,9.55
address_2,52082,9.55
square_feet,29669,5.44



**Initial Findings**

The missing value patterns in the complete Chicago dataset are consistent with those observed in the preliminary random sample.

The target variable, `rent_amount`, has no missing values. Most selected predictors are also complete or have relatively low missingness.

The variable `half_baths` has the highest proportion of missing values (82.62%) and will be excluded from the initial models.

We retain `year_built` (32.43% missing), `square_feet` (5.44% missing), and `full_baths` (0.03% missing). Missing values in these predictors will be handled through median imputation within the modeling pipelines.

This approach preserves observations while preventing information from the test set from influencing preprocessing.



### 3.2 Invalid Values Assessment

We examine the selected numerical variables for invalid or implausible values that could affect preliminary model training.

The checks focus on nonpositive rental prices and property sizes, negative bedroom or bathroom counts, and construction years outside a plausible range.

These checks are intended to identify potential data quality issues rather than automatically remove unusual observations.


In [10]:
# Check for invalid or implausible numerical values

invalid_checks = {
    "Nonpositive rent_amount":
        (chicago_df["rent_amount"] <= 0).sum(),

    "Nonpositive square_feet":
        (chicago_df["square_feet"] <= 0).sum(),

    "Negative bedrooms":
        (chicago_df["bedrooms"] < 0).sum(),

    "Negative full_baths":
        (chicago_df["full_baths"] < 0).sum(),

    "Negative half_baths":
        (chicago_df["half_baths"] < 0).sum(),

    "year_built outside 1800-2026":
        (
            chicago_df["year_built"].notna()
            & ~chicago_df["year_built"].between(1800, 2026)
        ).sum()
}

invalid_summary = pd.DataFrame(
    invalid_checks.items(),
    columns=["Check", "Invalid Records"]
)

display(invalid_summary)


,Check,Invalid Records
0,Nonpositive rent_amount,0
1,Nonpositive square_feet,1
2,Negative bedrooms,0
3,Negative full_baths,0
4,Negative half_baths,0
5,year_built outside 1800-2026,0



**Initial Findings**

Basic numerical validation identified only one record with nonpositive square footage among the 545,312 Chicago listings.

No nonpositive rental prices, negative bedroom or bathroom counts, or construction years outside the preliminary range of 1800–2026 were detected.

These results suggest that the selected numerical variables contain relatively few invalid values under the initial validation criteria.

The record with invalid square footage will be inspected before determining the appropriate treatment. Additional extreme values may still require further investigation during model refinement.


In [11]:
# Inspect the record with invalid square footage

invalid_sqft_records = chicago_df.loc[
    chicago_df["square_feet"].notna()
    & (chicago_df["square_feet"] <= 0)
]

display(invalid_sqft_records)


,dwellsy_address_id,address1_id,address2_id,data_source_external_id,listing_id,community_id,community_name,company_name,top_down_community_count,top_down_address1_count,...,ss_raw_secondary_designator,ss_raw_secondary_number,ss_raw_default_city_name,ss_raw_state_abbreviation,ss_raw_zipcode,ss_raw_plus4_code,amenities,photos,uru_id,uru_type
163544,9411189-31647770131,9411189,3.164777e+10,11148349,11922812,80255,1700-04 N Melvina Ave & 6204-08 W Wabansia Ave,Intendance,11,2,...,#,1,Chicago,IL,60639,3805,Certain Dog Breeds Not Allowed; No Cats Allowe...,https://media.dwellsy.com/prod/a15ee6694b3e75c...,4074440,Apartment



### 3.3 Treatment of Invalid Values

The data quality assessment identified one listing with `square_feet = 0` and one listing with an implausible value of `full_baths = 2201`.

Both values were replaced with missing values (`NaN`) rather than removing the observations. For `full_baths`, values greater than 10 were flagged as implausible, affecting only one observation in the dataset.

Missing values will be handled during model training using imputation procedures fitted exclusively on the training data.


In [12]:
# Replace invalid square footage values with NaN

chicago_df.loc[
    chicago_df["square_feet"] <= 0,
    "square_feet"
] = np.nan

# Verify the correction
print(
    "Nonpositive square_feet:",
    (chicago_df["square_feet"] <= 0).sum()
)

print(
    "Missing square_feet:",
    chicago_df["square_feet"].isna().sum()
)

print(
    "Total observations:",
    f"{len(chicago_df):,}"
)


Nonpositive square_feet: 0
Missing square_feet: 29670
Total observations: 545,312


In [13]:
# Replace implausible full bathroom counts with missing values
invalid_bath_mask = chicago_df["full_baths"] > 10

print("Invalid full bathroom counts:", invalid_bath_mask.sum())

chicago_df.loc[invalid_bath_mask, "full_baths"] = np.nan

print("Maximum full bathrooms after cleaning:", chicago_df["full_baths"].max())


Invalid full bathroom counts: 1
Maximum full bathrooms after cleaning: 9.0



## 4. Preparation of the Complete Dataset

This section prepares the complete Chicago dataset for subsequent rental price modeling.

All 47 original variables are retained. Two temporal features, `listing_year` and `listing_month`, are derived from `creation_time` to capture potential changes in rental market conditions.

Invalid square footage values have been treated as missing. Other missing values remain unchanged and will be handled within the appropriate modeling pipelines.

### 4.1 Temporal Feature Creation

We derive listing year and month from the listing creation timestamp without removing any original variables.

The variable `creation_time` represents the listing creation timestamp and does not necessarily indicate when the advertised rental price became effective.


In [14]:
# Create temporal features
chicago_df["listing_year"] = chicago_df["creation_time"].dt.year
chicago_df["listing_month"] = chicago_df["creation_time"].dt.month

# Preserve all original variables and derived features
chicago_prepared_df = chicago_df.copy()

print("Prepared dataset shape:", chicago_prepared_df.shape)

print("\nListing year range:")
print(
    chicago_prepared_df["listing_year"].min(),
    "-",
    chicago_prepared_df["listing_year"].max()
)

print("\nMissing temporal values:")
print(
    chicago_prepared_df[
        ["listing_year", "listing_month"]
    ].isna().sum()
)

display(chicago_prepared_df.head())


Prepared dataset shape: (545312, 49)

Listing year range:
2020 - 2026

Missing temporal values:
listing_year     0
listing_month    0
dtype: int64


,dwellsy_address_id,address1_id,address2_id,data_source_external_id,listing_id,community_id,community_name,company_name,top_down_community_count,top_down_address1_count,...,ss_raw_default_city_name,ss_raw_state_abbreviation,ss_raw_zipcode,ss_raw_plus4_code,amenities,photos,uru_id,uru_type,listing_year,listing_month
0,133822-8163780,133822,8163780.0,1739819,3675156,279577,6211 N Kenmore Ave,Mihail Marinov,28,28,...,Chicago,IL,60660,2151,12 Month Lease or Longer; Cable Ready; Cats OK...,https://media.dwellsy.com/prod/c151dd5f3919432...,542230,Apartment,2021,2
1,129521-8112041,129521,8112041.0,3871623,3558998,6069979,Mode Logan Square,Jessica Goodnight,296,218,...,Chicago,IL,60642,6024,12 Month Lease or Longer; Air Conditioning; Ca...,https://media.dwellsy.com/prod/8c186d97caa2102...,1362919,Apartment,2021,2
2,123813-8066896,123813,8066896.0,14582015,16668829,279329,21st Street Lofts,3L Living,137,137,...,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610,Apartment,2025,5
3,123813-8066896,123813,8066896.0,14582015,19878436,279329,21st Street Lofts,3L Living,137,137,...,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610,Apartment,2026,5
4,130630-8121250,130630,8121250.0,10139825,10639335,5764,2420 N Kedzie Blvd,Sam,54,54,...,Chicago,IL,60647,5740,Cats OK; Dishwasher; Gas included in rent; Har...,https://media.dwellsy.com/prod/cd5a80b0a1d4a17...,428523,Apartment,2023,6


In [15]:
# Assess geographic coordinate completeness and ranges
geo_columns = ["latitude", "longitude"]

print("Missing geographic values:")
print(chicago_prepared_df[geo_columns].isna().sum())

print("\nGeographic coordinate summary:")
display(chicago_prepared_df[geo_columns].describe())


Missing geographic values:
latitude     0
longitude    0
dtype: int64

Geographic coordinate summary:


,latitude,longitude
count,545312.000000,545312.000000
mean,41.912069,-87.704285
std,0.178235,0.203256
min,40.107159,-89.127007
25%,41.871590,-87.709709
50%,41.919300,-87.656586
75%,41.962357,-87.634369
max,42.743649,-86.766037



### 4.2 Saving the Complete Prepared Dataset

The prepared Chicago dataset retains all 47 original variables and includes two derived temporal features.

The dataset is saved for subsequent modeling and geospatial analysis. Feature selection and model-specific preprocessing will be performed in separate notebooks.


In [16]:
# Save the complete prepared Chicago dataset
paths.PROCESSED.mkdir(parents=True, exist_ok=True)
output_path = paths.PROCESSED / "chicago_prepared_full.csv"

chicago_prepared_df.to_csv(output_path, index=False)

print("Dataset saved to:", output_path)
print("Dataset shape:", chicago_prepared_df.shape)
print("File exists:", output_path.exists())


Dataset saved to: /Users/vivianaseleniahanampa/Columbia/Fall 2026/Capstone Project Cod/cleanup/data/processed/chicago_prepared_full.csv
Dataset shape: (545312, 49)
File exists: True


### 4.3 Final Validation


In [17]:
# Validate the complete prepared dataset
assert chicago_prepared_df.shape == (545312, 49)

assert chicago_prepared_df["uru_id"].notna().all()
assert chicago_prepared_df["rent_amount"].notna().all()
assert chicago_prepared_df["creation_time"].notna().all()

# Verify that all original columns are preserved
assert set(original_columns).issubset(
    chicago_prepared_df.columns
)

# Verify that geographic and temporal features are available
required_columns = [
    "latitude",
    "longitude",
    "listing_year",
    "listing_month"
]

assert all(
    col in chicago_prepared_df.columns
    for col in required_columns
)

print("All validation checks passed.")


All validation checks passed.
